# Tutorial 05: Memory Systems

**Time**: 30 minutes | **Difficulty**: Intermediate | **Prerequisites**: Tutorials 01-04

---

## What You'll Learn
- What Multiverse's central memory is and why it exists
- How experiences from one verse can speed up learning in another
- The difference between short-term memory (STM) and long-term memory (LTM)
- How to inspect and query the memory system

---

## The Problem: Learning from Scratch Every Time

Standard RL agents have **no memory between training runs**.
Train on `grid_world`, then switch to `maze_world` — the agent starts from zero.

But humans don't do this. If you've learned to navigate a grid, you already know something useful about navigating a maze.

Multiverse's central memory lets agents **accumulate experience across verses and training runs**.

```
Run 1: train on line_world → memories stored
Run 2: train on grid_world → memories stored, line_world memories available
Run 3: train on maze_world → can recall relevant spatial navigation memories
                             → learns faster than starting from scratch
```

### STM vs LTM

- **Short-Term Memory (STM)**: Recent experiences, high detail, fades over time
- **Long-Term Memory (LTM)**: Consolidated knowledge, promoted from STM based on value
- **Tier policy**: Rules for when to promote STM → LTM

---

## Setup

In [ ]:
import subprocess
import os
import json

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    output = result.stdout + result.stderr
    print(output if output.strip() else '(no output)')
    return result.returncode == 0

# Check memory system status
print('Checking Multiverse and memory system status...')
run('multiverse doctor')

## Part 1: Inspect the Central Memory

The central memory lives in `central_memory/`. Let's see what's there.

In [ ]:
import pathlib

memory_dir = pathlib.Path('central_memory')

if memory_dir.exists():
    files = list(memory_dir.iterdir())
    print(f'Central memory directory: {memory_dir.resolve()}')
    print(f'Files found: {len(files)}')
    for f in sorted(files):
        size = f.stat().st_size if f.is_file() else '-'
        print(f'  {f.name:40s}  {size} bytes')
else:
    print('No central_memory/ directory yet.')
    print('It will be created after your first memory-enabled training run.')

In [ ]:
# Peek at the memories file if it exists
memories_file = pathlib.Path('central_memory/memories.jsonl')

if memories_file.exists():
    with open(memories_file) as f:
        lines = f.readlines()
    
    print(f'Total memories stored: {len(lines)}')
    print()
    
    # Show a sample of memories
    sample = lines[:3]
    for i, line in enumerate(sample, 1):
        mem = json.loads(line)
        print(f'Memory {i}:')
        for key in ('verse', 'algo', 'episode_return', 'tags', 'tier'):
            if key in mem:
                print(f'  {key}: {mem[key]}')
        print()
else:
    print('No memories.jsonl yet. Run some training first!')

## Part 2: Training Without Memory (Baseline)

Train on `line_world`, then immediately train on `grid_world` with no memory.
Record the average return on grid_world.

In [ ]:
print('Step 1: Train on line_world (no memory transfer)')
run('multiverse train --algo q --verse line_world --episodes 100 --seed 5')

In [ ]:
print('Step 2: Train fresh on grid_world (no memory from line_world)')
run('multiverse train --algo q --verse grid_world --episodes 100 --seed 5')

## Part 3: Check What Was Stored

After training, the runs contain artifacts. Let's inspect the latest run.

In [ ]:
print('Latest run info:')
run('multiverse runs latest')

In [ ]:
print('Files in latest run:')
run('multiverse runs files latest')

## Part 4: Recall Lift — Measuring Transfer Benefit

Recall lift measures how much faster an agent learns when memory is available vs. starting from scratch.

A recall lift > 1.0 means memory helped. A value of 1.2 means the agent learned 20% faster.

In [ ]:
# Measure recall lift using the dedicated tool
print('Measuring recall lift...')
run('python tools/measure_recall_lift.py --help')

In [ ]:
# Run a preset recall lift benchmark
print('Running recall lift preset...')
run('python tools/run_recall_lift_preset.py')

## Part 5: Inspect the Tier Policy

The tier policy controls what gets promoted from STM to LTM.
By default, memories with high returns or novelty get promoted.

In [ ]:
tier_policy_file = pathlib.Path('central_memory/tier_policy.json')

if tier_policy_file.exists():
    with open(tier_policy_file) as f:
        policy = json.load(f)
    print('Tier policy:')
    print(json.dumps(policy, indent=2))
else:
    print('No tier_policy.json found.')
    print('Create one to control STM → LTM promotion rules.')
    print()
    print('Example tier policy:')
    example = {
        'promote_threshold': 0.7,
        'novelty_weight': 0.3,
        'return_weight': 0.7,
        'max_ltm_entries': 10000
    }
    print(json.dumps(example, indent=2))

## What You Learned

| Concept | What It Means |
|---|---|
| Central memory | Persistent store of experiences across training runs |
| STM | Short-term: recent, detailed, temporary |
| LTM | Long-term: consolidated, promoted based on value/novelty |
| Recall lift | How much memory accelerates learning on a new task |
| Tier policy | Rules for promoting STM → LTM |

Memory becomes most valuable when:
- You train on many related verses (transfer is meaningful)
- Individual training runs are expensive
- You want agents that accumulate expertise over time

---

## Next

- [Tutorial 06: Custom Verses — build your own environment](06_custom_verses.ipynb)